In [2]:
# run_real_binance.py
from __future__ import annotations
import numpy as np
from datetime import datetime, timezone

from binance_data import fetch_book_tickers, fetch_klines
from dtw_similarity import build_daily_sequences_from_hourly_klines, compute_similarity_matrix
from market_graph import build_market_weights
from pair_search import find_multiple_pairs

def utc_day_from_ms(ts_ms: int) -> int:
    return int(datetime.fromtimestamp(ts_ms / 1000.0, tz=timezone.utc).strftime("%Y%m%d"))

def fetch_base_open_today(symbol: str) -> float:
    """
    Берём open текущей UTC-дневной свечи (interval=1d).
    Это и есть base price on the day (в духе статьи).
    """
    k1d = fetch_klines(symbol, interval="1d", limit=2)  # usually [yesterday, today]
    if not k1d:
        raise RuntimeError(f"No 1d klines for {symbol}")

    # last candle is "today" (ongoing), its open is base for today
    base_open = float(k1d[-1][1])
    if base_open <= 0:
        raise RuntimeError(f"Bad base_open for {symbol}: {base_open}")
    return base_open

symbols = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT", "ADAUSDT"]

# 1) best bid/ask сейчас
ba = fetch_book_tickers(symbols)

# 2) 1h klines for ~8 days
daily_seq = {}
for sym in symbols:
    kl_1h = fetch_klines(sym, interval="1h", limit=220)  # ~9 days
    daily_seq[sym] = build_daily_sequences_from_hourly_klines(kl_1h)

# 3) similarity по последним 5 полным дням (исключаем "сегодня")
S, used_days = compute_similarity_matrix(symbols, daily_seq, days_back=5, exclude_last_day=True)
print("Similarity days used (UTC):", used_days)

# 4) нормируем ask/bid на base_open текущего дня (по статье)
ask = np.zeros(len(symbols), dtype=float)
bid = np.zeros(len(symbols), dtype=float)

for i, sym in enumerate(symbols):
    base_open = fetch_base_open_today(sym)
    b, a = ba[sym]
    bid[i] = b / base_open
    ask[i] = a / base_open

# 5) строим веса w (с dummy=0)
w = build_market_weights(ask=ask, bid=bid, sim=S)

# 6) поиск пар
pairs = find_multiple_pairs(
    w=w,
    threshold=-0.002,      # начни мягко, потом калибруй
    max_pairs=10,
    sb_variant="BSB",
    n_runs_per_pair=200,
    n_iter=500,
    dt=1.0,
    mp=200.0,
    seed=42,
    debug=True,
)

print("\n=== RESULTS ===")
for sol in pairs:
    short_sym = symbols[sol.short_node - 1]
    long_sym  = symbols[sol.long_node - 1]

    direct_w = float(w[sol.short_node, sol.long_node])
    kind = "BYPASS" if len(sol.path_nodes) > 2 else "DIRECT"
    path_syms = [symbols[n - 1] for n in sol.path_nodes]

    print(f"{kind}: short={short_sym} long={long_sym}  "
            f"path={' -> '.join(path_syms)}  "
            f"path_weight={sol.path_weight:.6g}  direct_w={direct_w:.6g}")

Similarity days used (UTC): [20260224, 20260225, 20260226, 20260227, 20260228]
[k=0] best valid cycle weight=-0.03195980124204614
[k=1] best valid cycle weight=-0.022168904508874383
[k=2] best valid cycle weight=-0.02145715628668581
[k=3] best valid cycle weight=-0.02547788221260488
[k=4] best valid cycle weight=-0.02097844487203235
[k=5] best valid cycle weight=-0.01978832832886363
[k=6] best valid cycle weight=-0.019571056762033554
[k=7] best valid cycle weight=-0.01901117437544394
[k=8] best valid cycle weight=-0.020053792618379775
[k=9] best valid cycle weight=-0.017137717027525385

=== RESULTS ===
BYPASS: short=ETHUSDT long=BTCUSDT  path=ETHUSDT -> XRPUSDT -> SOLUSDT -> ADAUSDT -> BNBUSDT -> BTCUSDT  path_weight=-0.0319598  direct_w=-0.00979381
BYPASS: short=ETHUSDT long=SOLUSDT  path=ETHUSDT -> ADAUSDT -> BNBUSDT -> XRPUSDT -> BTCUSDT -> SOLUSDT  path_weight=-0.0221689  direct_w=-0.00580956
BYPASS: short=ETHUSDT long=ADAUSDT  path=ETHUSDT -> SOLUSDT -> BNBUSDT -> BTCUSDT -> XRPUS

In [3]:
from verify_bruteforce import verify_solutions
verify_solutions(w, pairs)

Exact global best: (2, 1) value= -0.03311524001996084

[k=0] short=2 long=1
  reported: -0.03195980124204614
  recomputed: -0.03195980124204614 diff= 0.0
  exact min simple-path: -0.03311524001996084 gap= 0.0011554387779146996

[k=1] short=2 long=4
  reported: -0.022168904508874383
  recomputed: -0.022168904508874383 diff= 0.0
  exact min simple-path: -0.02888789026678186 gap= 0.006718985757907477

[k=2] short=2 long=6
  reported: -0.02145715628668581
  recomputed: -0.02145715628668581 diff= 0.0
  exact min simple-path: -0.025620530652503182 gap= 0.004163374365817374

[k=3] short=4 long=1
  reported: -0.02547788221260488
  recomputed: -0.02547788221260488 diff= 0.0
  exact min simple-path: -0.02730568127921343 gap= 0.00182779906660855

[k=4] short=3 long=5
  reported: -0.02097844487203235
  recomputed: -0.02097844487203235 diff= 0.0
  exact min simple-path: -0.023940550483902385 gap= 0.0029621056118700334

[k=5] short=5 long=6
  reported: -0.01978832832886363
  recomputed: -0.019788328

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_spins_heatmap(traj, every=1, use_sign=True, title="spin trajectories"):
    """
    traj: (T, N) — x[t,i]
    use_sign=True -> рисуем sign(x) как спины {-1,0,+1}
    """
    data = traj[::every]
    if use_sign:
        data = np.sign(data)

    plt.figure()
    plt.imshow(data.T, aspect="auto", origin="lower")
    plt.xlabel("time step")
    plt.ylabel("spin index")
    plt.title(title)
    plt.colorbar()
    plt.show()